# Reto Tokio / GCI World NFL Draft Prediction
## Final Integrated Project Report

**Author:** Jonatan Estiven Sanchez Vargas
**Institution:** Universidad Nacional de Colombia
**Program:** Systems and Computer Engineering (Ingeniería de Sistemas e Informática)
**Project:** Reto Tokio / GCI World NFL Draft Prediction
**Task:** Binary classification — predict the probability that an athlete is drafted (`Drafted = 1`)
**Official metric:** ROC-AUC on the positive-class probability

---

This notebook is an *executable technical report*. It tells the complete story of the project — from
the competition contract through eleven gated phases to two validated, submission-ready prediction
files — and it can be run top-to-bottom in three environments (a full repository, this portable
package, or Google Colab with only the three official CSV files).

The work prioritises **reproducibility, leakage safety, and auditability** over leaderboard chasing.
No single model is declared "the winner": the project produces a primary candidate and a stable
fallback, validates both, and leaves the upload decision as a deliberate manual choice.

**Provenance labels.** Every quantitative result below carries one of four labels so the reader always
knows how a number was obtained:

| Label | Meaning |
|---|---|
| **Computed in this notebook** | Produced by code executing in the current runtime. |
| **Loaded from repository artifact** | Read from a persisted file when running inside the full repository. |
| **Recorded from accepted project artifact** | Quoted from an accepted phase result; not recomputed here. |
| **Not available in current runtime** | Cannot be produced in the active mode; the recorded value is shown. |


## Executive summary

The competition asks for the probability that a college athlete is selected in the NFL Draft, scored
by **ROC-AUC on the positive-class probability**. The training set has 2,781 rows and the test set
has 696 rows; the submission must contain exactly **696 rows** in the column format `Id,Drafted`, with
the `Id` order matching `sample_submission.csv`. The positive class is the **majority** class
(positive rate ≈ **0.6483**), which is why the project optimises a threshold-free ranking metric rather
than accuracy.

The solution was built under a strict, phase-gated discipline:

- A single **frozen 5-fold split** (`StratifiedKFold(5, shuffle=True, random_state=42)`) is shared by
  every model comparison, and **all learned preprocessing is fitted inside the training folds** (or on
  the full training set at the final refit) — never on test data.
- The model uses a **21-feature set ("F2")** that includes the raw measurements, **explicit missingness
  flags**, and an **availability count**, but excludes `School` (high-cardinality leakage risk) and uses
  no external data.
- Two candidates reached submission readiness and **no winner is declared**:

| Candidate | Role | OOF ROC-AUC | Provenance |
|---|---|---:|---|
| Tuned CatBoost | Primary (warning-heavy) | 0.830321 | Recorded from accepted project artifact |
| Logistic regression (baseline) | Fallback / reference | 0.827082 | Recorded from accepted project artifact |
| RandomForest (frozen) | Anchor / reference | 0.811650 | Recorded from accepted project artifact |

Both candidates were refit on the full training set and produced format-valid 696-row submissions. The
notebook below reproduces the logistic-regression candidate live (and the tuned CatBoost candidate when
the optional `catboost` package is available), validates the result against the official contract, and
records the SHA-256 identity of each file. **The choice of which file to submit, and in what order, is
a manual decision; the last submitted file determines the final ranking.**

#### Section takeaways
- The objective is probability ranking quality (ROC-AUC), under a majority-positive class balance.
- The pipeline is leakage-safe and frozen-fold; two validated candidates exist; the upload decision is manual.


## How to run this notebook in Google Colab

This notebook detects its environment and adapts. There are three supported modes:

| Mode | Data source | Models computed live | Heavy historical evidence |
|---|---|---|---|
| **Full repository** | `data/input/` + stored artifacts | RandomForest anchor, logistic regression | Loaded from stored artifacts |
| **Portable package** | `data/input/` (this package) | Logistic regression always; CatBoost if `catboost` is installed | Recorded values |
| **Colab minimal** | three uploaded CSVs | Logistic regression always | Recorded values |

**Steps in Colab:**

1. Upload this package folder to Google Drive and open `99_final_integrated_project_report.ipynb`,
   **or** open the notebook and upload only `train.csv`, `test.csv`, and `sample_submission.csv`
   when prompted by the optional upload cell.
2. Choose **Runtime → Run all**.
3. The notebook resolves its own paths, rebuilds the validation and feature pipeline, refits the
   reproducible candidate, writes a validated submission to `outputs/submissions/`, and prints its
   SHA-256.

The notebook needs only **pandas, numpy, and scikit-learn**. CatBoost is optional: if it is not
installed, the tuned-CatBoost figures are shown as recorded values and the notebook proceeds with the
logistic-regression candidate, which still yields a fully validated submission.

#### Section takeaways
- One notebook, three runtime modes; no manual path edits are required.
- The minimal dependency set always produces a valid 696-row submission; CatBoost is an optional extra.


### Environment, path configuration, and seeds

The cell below establishes a portable project root, resolves the input/output locations relative to it,
fixes the random seed, and performs *soft* imports so the notebook degrades gracefully when an optional
dependency is missing. It writes no data and trains nothing; it only prepares the runtime.

In [ ]:
# Environment detection, robust path resolution, seeds, and soft imports.
from pathlib import Path
import sys, os, math, json, hashlib, importlib

PROJECT_SEED = 42

def _is_colab():
    return ("google.colab" in sys.modules) or Path("/content").exists()

IS_COLAB = _is_colab()

def _find_project_root():
    """Locate a directory that contains data/input/train.csv across local/package/Colab layouts."""
    here = Path.cwd()
    candidates = [here, *here.parents,
                  here / "final_integrated_notebook_package",
                  Path("/content"), Path("/content/final_integrated_notebook_package"),
                  Path("/content/drive/MyDrive"),
                  Path("/content/drive/MyDrive/final_integrated_notebook_package")]
    for c in candidates:
        try:
            if (c / "data" / "input" / "train.csv").is_file():
                return c
        except Exception:
            pass
    return here  # fall back to cwd; the upload cell can populate it

PROJECT_ROOT = _find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "input"
OUT_DIR = PROJECT_ROOT / "outputs"
for sub in ["submissions", "folds", "figures", "tables"]:
    (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)

# Required dependencies.
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

np.random.seed(PROJECT_SEED)

# Optional dependencies (soft).
def _try_import(name):
    try:
        return importlib.import_module(name)
    except Exception:
        return None

plt = _try_import("matplotlib.pyplot")
HAS_MPL = plt is not None
catboost = _try_import("catboost")
HAS_CATBOOST = catboost is not None

def pkg_version(name):
    try:
        return importlib.metadata.version(name)
    except Exception:
        return "not installed"

print("Runtime configuration")
print("  IS_COLAB        :", IS_COLAB)
print("  PROJECT_ROOT    :", PROJECT_ROOT)
print("  data dir exists :", DATA_DIR.is_dir())
print("  matplotlib      :", "available" if HAS_MPL else "absent (figures skipped)")
print("  catboost        :", "available" if HAS_CATBOOST else "absent (logistic-regression fallback)")
print("  versions        : pandas", pkg_version("pandas"),
      "| numpy", pkg_version("numpy"), "| scikit-learn", pkg_version("scikit-learn"))


### Optional: upload the official files in Colab

Run the next cell **only** if the three official CSV files were not found automatically (it is a no-op
outside Colab and when the files already exist). It opens the Colab upload dialog and moves the files
into `data/input/`.

In [ ]:
# Optional Colab upload fallback. Safe no-op outside Colab / when files already present.
def _inputs_present():
    return all((DATA_DIR / f).is_file() for f in ["train.csv", "test.csv", "sample_submission.csv"])

if IS_COLAB and not _inputs_present():
    try:
        from google.colab import files
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        print("Please upload train.csv, test.csv and sample_submission.csv ...")
        uploaded = files.upload()
        for name in uploaded:
            target = DATA_DIR / Path(name).name
            Path(name).replace(target)
        print("Stored:", [p.name for p in DATA_DIR.glob("*.csv")])
    except Exception as exc:
        print("Upload dialog unavailable:", exc)
else:
    print("Inputs present:", _inputs_present(), "(no upload needed)")


## Competition objective and machine-learning framing

The GCI World NFL Draft Prediction challenge is a **binary classification** problem: for each athlete,
estimate the probability of `Drafted = 1`. Submissions are ranked by **ROC-AUC computed on the
positive-class probability**, so the model must produce well-ordered probabilities rather than hard
labels.

**Why ROC-AUC, and why probabilities.** The positive class is the majority (positive rate ≈ 0.6483),
and the official metric is a ranking metric. Accuracy at a fixed 0.5 threshold would be both
metric-mismatched and sensitive to the imbalance direction. ROC-AUC on probabilities measures how well
the model separates drafted from non-drafted athletes across all thresholds, which is exactly what the
competition rewards. Throughout, the positive-class probability is extracted **only after verifying that
the fitted estimator's `classes_` contains the label `1`**, never by blind column indexing.

**Data policy.** Only the official competition files are used. No external athlete, school, conference,
ranking, geography, draft-history, or online-statistics data is introduced anywhere; external data would
invalidate the ranking.

#### Section takeaways
- The task is probability ranking quality, scored by ROC-AUC, under a majority-positive class balance.
- Positive-class probabilities are always extracted via a verified `classes_` index.


## Official data contract

The next cell loads the three official files and verifies the contract live: shapes, target presence,
class balance, identifier alignment, and per-column missingness. These numbers are **Computed in this
notebook**.

In [ ]:
# Load official files and verify the data contract (computed live).
assert _inputs_present(), f"Official CSVs not found under {DATA_DIR}. Run the upload cell or place them there."

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")

ID_COL, TARGET = "Id", "Drafted"

print("Shapes  -> train:", train.shape, "| test:", test.shape, "| sample_submission:", sample.shape)
assert train.shape == (2781, 16), "Unexpected train shape"
assert test.shape == (696, 15), "Unexpected test shape"
assert sample.shape == (696, 2), "Unexpected sample_submission shape"
assert TARGET in train.columns and TARGET not in test.columns, "Target column contract violated"
assert list(sample.columns) == [ID_COL, TARGET], "sample_submission columns must be exactly [Id, Drafted]"

# Class balance and positive rate.
counts = train[TARGET].value_counts().sort_index()
pos_rate = train[TARGET].mean()
print("\nClass balance -> 0:", int(counts.get(0, 0)), "| 1:", int(counts.get(1, 0)),
      "| positive rate: %.10f" % pos_rate)

# Id alignment between test and sample_submission.
print("Id order test == sample_submission:", bool((test[ID_COL].values == sample[ID_COL].values).all()))

# Missingness summary (train vs test).
miss_cols = ["Age", "Sprint_40yd", "Vertical_Jump", "Bench_Press_Reps",
             "Broad_Jump", "Agility_3cone", "Shuttle"]
miss = pd.DataFrame({
    "train_missing": [int(train[c].isna().sum()) for c in miss_cols],
    "test_missing":  [int(test[c].isna().sum()) for c in miss_cols],
}, index=miss_cols)
print("\nMissingness (rows):")
print(miss.to_string())


**Reading the contract.** The shapes (2781×16 / 696×15 / 696×2), the class balance
(978 negative / 1,803 positive, positive rate ≈ 0.6483), and the `Id` alignment all hold. The
missingness is substantial and *structured*: drills such as `Agility_3cone` and `Shuttle` are missing
for a large share of athletes, and the test set shows a similar pattern (about one in six test rows has
a missing `Age`). This structure is the central modelling signal developed in Phase 7. The test set has
no `Drafted` column and is never used for fitting — only for the final inference.

#### Section takeaways
- The official contract holds exactly; the data is majority-positive with structured missingness.
- Missingness is informative and will be modelled explicitly rather than silently imputed away.


### Optional exploratory figures

If `matplotlib` is available, the next cell draws two small, fast figures (target distribution and
train/test missingness). They are illustrative; the full exploratory set was produced during Phase 3
and is recorded there.

In [ ]:
# Optional, fast EDA figures (skipped cleanly if matplotlib is absent).
if HAS_MPL:
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    counts.plot(kind="bar", ax=ax[0], color=["#b0413e", "#3b7a57"])
    ax[0].set_title("Target distribution (train)"); ax[0].set_xlabel("Drafted"); ax[0].set_ylabel("rows")
    miss.plot(kind="bar", ax=ax[1])
    ax[1].set_title("Missing rows by column"); ax[1].set_ylabel("rows")
    plt.tight_layout()
    fig.savefig(OUT_DIR / "figures" / "eda_overview.png", dpi=110)
    plt.show()
    print("Saved:", OUT_DIR / "figures" / "eda_overview.png")
else:
    print("matplotlib not available — figures skipped (tables above carry the same information).")


## Phase 1 — Project contract and rules

**Purpose.** Establish the working discipline before any modelling: a notebook-first workflow,
reproducible-from-a-clean-kernel execution, fixed random seeds, the use of official files only, and
strict version control of which artifacts may change.

**Decisions and rationale.** The project committed early to: deterministic seeds (the value `42` for
fold generation); a prohibition on editing predictions by hand; the rule that the public leaderboard is
a sanity check, never a selection tool; and the exclusion of any external data. These rules exist
because the competition explicitly disqualifies non-reproducible or externally-augmented entries
regardless of leaderboard position — so reproducibility is treated as a correctness requirement, not a
nicety.

*Transition.* With governance, seeds, and the official-files-only rule fixed, the project moves to
reproducing the documented baseline as a historical reference.

#### Key conclusions from this phase
- A reproducible, auditable, official-data-only workflow was established before any modelling.
- Phase 1 was never formally "closed" in the early history; its rules were carried forward implicitly
  and later codified — noted here for honesty rather than overstated. *(Recorded from accepted project
  artifact.)*


## Phase 2 — Baseline reproduction

**Purpose.** Reproduce the official baseline to obtain a historical reference point and confirm the
data pipeline end-to-end.

**Result.** The official baseline — a `RandomForestClassifier(n_estimators=100, max_depth=5,
random_state=2025)` with global mean imputation, label encoding, and an engineered BMI feature —
reproduces at:

| Quantity | Value | Provenance |
|---|---:|---|
| Cross-validated ROC-AUC | 0.812964 ± 0.025740 | Recorded from accepted project artifact |
| Public leaderboard AUC | 0.80792 | Recorded from accepted project artifact |

**Rationale and caveat.** This baseline is **leakage-inflated by design**: its imputation, encoding, and
BMI are fitted on the whole dataset rather than inside cross-validation folds. It is therefore retained
strictly as a *historical reference*, not as the project's anchor. The later clean, fold-safe pipeline
scores lower out-of-fold precisely because it removes this optimistic bias — an expected correction,
not a regression.

*Transition.* Because the reproduced baseline is leakage-inflated by design, it is treated as reference
only, and the project shifts to leakage-aware exploratory analysis that locks no feature.

#### Key conclusions from this phase
- The official baseline was reproduced and recorded as a reference (CV ROC-AUC 0.812964; public LB 0.80792).
- Its global preprocessing is leakage-prone; it does not serve as the clean anchor.


## Phase 3 — Exploratory data analysis and the data contract

**Purpose.** Map the signal families and risks and form hypotheses, without locking any feature.

**Findings.** Four hypothesis families emerged: (1) **role context** — draft rates vary strongly by
position and player type; (2) **measurement availability** — whether a drill was recorded is itself
informative; (3) **role-aware physical profile** — the meaning of a measurement depends on the role;
and (4) **institutional/categorical context** — `School` carries apparent signal but is high-cardinality
and includes test-only categories. The exploratory pass also confirmed the data contract (2781×16 /
696×15 / 696×2) and the positive rate of 0.6483279396, and produced two dozen figures.

**Decision and rationale.** A striking train-only association was noted: rows with missing `Age` show a
very different draft rate. This is a *hypothesis*, not a feature — adopting it would require fold-safe
validation, which is deferred to Phase 7. Diagnostic-only variables (such as a school-frequency group)
are explicitly kept out of the model. `School` is flagged as a leakage/instability risk.

*Transition.* The four hypothesised signal families and their risks are carried into a literature
synthesis that turns observations into enforceable rules.

#### Key conclusions from this phase
- Four signal families were hypothesised; nothing was locked as a feature.
- Missingness looks informative, but is adopted only later under fold-safe validation; `School` is flagged risky.


## Phase 4 — Research and methodological foundation

**Purpose.** Convert the literature and the EDA observations into enforceable project rules.

**Outcome.** The reference library (see `references.md`) was distilled into concrete rules:

| Rule | Supporting source (see references.md) |
|---|---|
| Fit every learned transform inside training folds; never touch test in fitting | Kapoor & Narayanan; *On leakage in ML pipelines* |
| Guard against model-selection overfitting; out-of-fold evaluation; no leaderboard for selection | Cawley & Talbot (2010) |
| Model missingness explicitly; evaluate features by slice | Kuhn & Johnson (Feature Engineering and Selection) |
| Compare gradient-boosted trees fairly before adopting them | Chen & Guestrin (2016); Ke et al. (2017); Prokhorenkova et al. (2018) |
| Keep hyperparameter search bounded and pre-registered | Akiba et al. (2019) |

**Rationale.** Each rule is grounded in a real source and is enforced mechanically in later phases (for
example, the fit-scope rule is implemented by fitting preprocessing inside each fold). On any conflict
between a secondary summary and the official competition brief, the brief governs.

*Transition.* The synthesised rules are formalised into a frozen methodology that binds every subsequent
phase.

#### Key conclusions from this phase
- The literature was translated into enforceable rules for validation, leakage, features, models, and HPO.
- Each rule maps to a real reference and is enforced mechanically later.


## Phase 5 — Frozen execution decisions

**Purpose.** Freeze the methodology so that every later phase is judged on identical terms.

**Frozen decisions.**

| Decision | Value |
|---|---|
| Metric | ROC-AUC on positive-class probability (via verified `classes_`) |
| Cross-validation | `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` |
| Comparison currency | Out-of-fold (OOF) ROC-AUC on a single frozen fold assignment |
| Fit scope | All learned transforms fitted inside training folds (full train at final refit) |
| `School` | Excluded as a feature (diagnostic only) until explicitly justified |
| Submissions | Permitted only in Phase 11, from a logged, reproducible candidate |
| Experiment log | Protected; candidate results recorded separately |

**Rationale.** Freezing the protocol prevents the subtle bias of changing the evaluation after seeing
results. A few quantitative thresholds (the feature-adoption margin, the minimum slice size, the
unit-of-observation question) were intentionally deferred to Phase 6A.

*Transition.* The frozen protocol is implemented as a leakage-safe validation harness producing the
canonical out-of-fold anchor on frozen folds.

#### Key conclusions from this phase
- The full evaluation protocol was frozen, removing post-hoc evaluation bias.
- Some quantitative thresholds remained deferred to Phase 6A *(noted as "Not confirmed yet" at freeze time)*.


## Phase 6 — Validation harness

**Purpose.** Implement the frozen protocol as a leakage-safe harness: a single frozen fold assignment,
out-of-fold predictions, and slice diagnostics.

**Result.** A clean OOF anchor was established on the frozen folds:

| Quantity | Value | Provenance |
|---|---:|---|
| Per-fold ROC-AUC | 0.690076 / 0.751758 / 0.761581 / 0.704939 / 0.737911 | Recorded from accepted project artifact |
| Fold mean ± std | 0.729253 ± 0.030629 | Recorded from accepted project artifact |
| OOF ROC-AUC (anchor) | 0.726616 | Recorded from accepted project artifact |
| Frozen fold file SHA256[:16] | 96937649526bcadb (2781 rows, folds 0..4) | Recorded from accepted project artifact |

**Rationale.** The project anchors on the **OOF** score (0.726616), not the fold mean (0.729253),
because the OOF score pools every validation prediction once and is the quantity later models are
compared against. The cell below **regenerates the fold assignment live** under the frozen protocol so
the reader can confirm the split is deterministic and stratified.

*Transition.* With a clean anchor in hand, the gap to the historical baseline is decomposed and the
ablation threshold and duplication verdict are established.

In [ ]:
# Regenerate the frozen 5-fold assignment (deterministic) and report it (computed live).
y = train[TARGET].astype(int).values
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=PROJECT_SEED)
fold_assignment = np.full(len(train), -1, dtype=int)
for fold_idx, (_, val_idx) in enumerate(skf.split(train, y)):
    fold_assignment[val_idx] = fold_idx
train = train.copy()
train["fold"] = fold_assignment

print("Fold sizes:", dict(pd.Series(fold_assignment).value_counts().sort_index()))
print("Per-fold positive rate:")
for f in range(5):
    m = fold_assignment == f
    print(f"  fold {f}: n={m.sum():4d}  positive_rate={y[m].mean():.4f}")
assert (fold_assignment >= 0).all() and set(np.unique(fold_assignment)) == {0,1,2,3,4}

# If the canonical fold file is reachable (full-repository mode), confirm its identity.
canonical = PROJECT_ROOT / "outputs" / "folds" / "phase6_rf_sanity_baseline_v1_fold_assignments.csv"
if canonical.is_file():
    sha16 = hashlib.sha256(canonical.read_bytes()).hexdigest()[:16]
    print(f"\nCanonical fold file present; SHA256[:16] = {sha16} (expected 96937649526bcadb) "
          f"-> {'MATCH' if sha16 == '96937649526bcadb' else 'DIFFERS'}  [Loaded from repository artifact]")
else:
    print("\nCanonical fold file not shipped in this package; folds regenerated deterministically "
          "(recorded identity: SHA256[:16] = 96937649526bcadb). [Recorded from accepted project artifact]")


#### Key conclusions from this phase
- A clean OOF anchor (0.726616) was established on a deterministic, stratified, frozen 5-fold split.
- The project compares models on the OOF score, not the fold mean; the fold identity is fixed.


## Phase 6A — Baseline reconciliation

**Purpose.** Explain the gap between the leakage-inflated Phase 2 baseline and the clean Phase 6 anchor,
and set two governance constants: the feature-adoption threshold and the duplication verdict.

**Findings.**

| Quantity | Value | Provenance |
|---|---:|---|
| Clean anchor (V0) OOF | 0.726616 | Recorded from accepted project artifact |
| Mean-imputation variant (V7) OOF | 0.802271 | Recorded from accepted project artifact |
| Seed-sweep OOF std (noise floor) | 0.002718 | Recorded from accepted project artifact |
| Feature-adoption threshold | OOF gain ≥ 0.005436 **and** same-sign in ≥ 4/5 folds | Recorded from accepted project artifact |
| Duplication escalation (strict tier) | 0% — cleared | Recorded from accepted project artifact |

**Rationale.** Most of the gap is explained by **informative missingness**: replacing in-fold median
imputation with mean imputation recovers much of the score, which signals that *the pattern of which
measurements are present* carries signal. This is not silently adopted; instead it motivates explicit
missingness features in Phase 7. The seed sweep fixes a noise floor, from which the adoption threshold
(≈ 2× the seed std) is derived. A duplication probe (possible athlete repetition across years) was run
and **cleared** at the strict tier, so grouped cross-validation stayed dormant. The minimum slice size
was set to n ≥ 50.

*Transition.* Having attributed most of the gap to informative missingness, the project tests
missingness and availability features explicitly and fold-safely.

#### Key conclusions from this phase
- The Phase 2↔6 gap is mostly informative missingness, motivating explicit missingness features (not silent mean imputation).
- Two constants were fixed: adoption threshold (≥ 0.005436 and 4/5 folds) and a cleared duplication verdict (no grouped CV).


## Phase 7 — Feature engineering: missingness and availability

**Purpose.** Test explicit missingness/availability features under fixed-fold ablation and adopt the
final feature set.

**Result.** Several variants were evaluated against the F0 base under the adoption rule:

| Variant | OOF ROC-AUC | Decision | Provenance |
|---|---:|---|---|
| F0 (base only) | 0.726616 | anchor | Recorded from accepted project artifact |
| F1 (base + flags) | 0.811568 | escalated (slice guard) | Recorded from accepted project artifact |
| **F2 (base + flags + availability count)** | **0.811650** | **adopted** | Recorded from accepted project artifact |
| F3 (F2 + completeness bins) | 0.810606 | escalated | Recorded from accepted project artifact |
| F5 (mean-impute + flags) | 0.813066 | escalated | Recorded from accepted project artifact |

**Decision and rationale.** **F2** was adopted by rule: its OOF gain over F0 (+0.0850) far exceeds the
0.005436 threshold, it improves in **5/5 folds**, and its mandatory-slice guard is clear. The
neighbouring variants (F1, F3, F5) showed larger losses on the fragile `Age_missing = 1` slice and so
were escalated rather than adopted. The project keeps **median imputation** (mean imputation is treated
as a diagnostic), models missingness with **explicit flags** rather than letting imputation encode it
implicitly, and continues to **exclude `School`** and BMI.

The cell below **builds F2 live** from the official files. The flags and the availability count are pure
row-wise transforms (they use only the current row), so they are leakage-safe by construction; the
imputation, scaling, and one-hot encoding are applied later **inside** the cross-validation pipeline.

*Transition.* With F2 adopted, the single deferred role interaction is probed under the same adoption
rule.

In [ ]:
# Build the F2 feature set (21 features) — row-wise, leakage-safe by construction.
BASE_NUMERIC = ["Year", "Age", "Height", "Weight", "Sprint_40yd", "Vertical_Jump",
                "Bench_Press_Reps", "Broad_Jump", "Agility_3cone", "Shuttle"]
CATEGORICAL = ["Player_Type", "Position_Type", "Position"]
PHYSICAL_TEST_COLUMNS = ["Sprint_40yd", "Vertical_Jump", "Bench_Press_Reps",
                         "Broad_Jump", "Agility_3cone", "Shuttle"]      # 6 drills -> count 0..6
FLAG_SOURCES = ["Age"] + PHYSICAL_TEST_COLUMNS                          # 7 missingness flags

def build_f2(df):
    """Return an F2 design frame: 13 base + 7 missingness flags + availability count (21 features)."""
    out = df.copy()
    for col in FLAG_SOURCES:
        out[f"{col}_missing"] = out[col].isna().astype(int)
    out["available_measurement_count"] = out[PHYSICAL_TEST_COLUMNS].notna().sum(axis=1).astype(int)
    flags = [f"{c}_missing" for c in FLAG_SOURCES]
    feature_cols = BASE_NUMERIC + CATEGORICAL + flags + ["available_measurement_count"]
    return out[feature_cols], feature_cols

X_train_df, F2_FEATURES = build_f2(train)
X_test_df, _ = build_f2(test)
NUMERIC_FEATURES = [c for c in F2_FEATURES if c not in CATEGORICAL]   # 18 numeric (incl. flags + count)

assert "School" not in F2_FEATURES, "School must never be a model feature"
assert len(F2_FEATURES) == 21, f"F2 must have 21 features, got {len(F2_FEATURES)}"
assert X_train_df["available_measurement_count"].between(0, 6).all(), "availability count out of 0..6"

print("F2 feature count:", len(F2_FEATURES), "(", len(NUMERIC_FEATURES), "numeric +", len(CATEGORICAL), "categorical )")
print("School excluded from features:", "School" not in F2_FEATURES)
print("available_measurement_count distribution:",
      dict(X_train_df["available_measurement_count"].value_counts().sort_index()))


#### Key conclusions from this phase
- F2 (21 features) was adopted by rule (+0.0850 OOF over F0, 5/5 folds, slice guard clear).
- Missingness is modelled with explicit flags; median imputation is kept; `School` and BMI stay out.


## Phase 7B — Role-aware interaction probe

**Purpose.** Evaluate the one deferred role interaction (availability count × player type) under the
same adoption rule.

**Result.**

| Variant | OOF ROC-AUC | Same-sign folds | Decision | Provenance |
|---|---:|---:|---|---|
| F2 | 0.8116502602 | — | retained | Recorded from accepted project artifact |
| F4 (F2 + role interaction) | 0.8093690702 | 1/5 | rejected | Recorded from accepted project artifact |

**Decision and rationale.** F4 was **rejected**: its OOF score is *below* F2 (−0.0023) and it improves in
only 1/5 folds, far short of the 4/5 requirement. Localised gains on small slices (special teams, kicking
specialists) did not survive the global out-of-fold evaluation. **F2 stands as the final 21-feature set.**

*Transition.* After the role interaction is rejected and F2 confirmed, the fixed feature set enters a
fair, frozen-fold model-family comparison.

#### Key conclusions from this phase
- The deferred role interaction (F4) was rejected; small-slice gains did not generalise out-of-fold.
- F2 is confirmed as the final, frozen feature set for all model comparisons.


## Phase 8 — Model-family comparison (scikit-learn native)

**Purpose.** Compare five scikit-learn models on the frozen F2 features and frozen folds, under
identical, fold-safe preprocessing.

**Result.**

| Model | OOF ROC-AUC | Folds vs anchor | Decision | Provenance |
|---|---:|---:|---|---|
| M0 RandomForest (frozen) | 0.8116502602 | — | anchor | Recorded from accepted project artifact |
| **M1 LogisticRegression** | **0.8270821070** | 4/5 | candidate-with-warning | Recorded from accepted project artifact |
| M2 RandomForest (default) | 0.8055237975 | — | rejected | Recorded from accepted project artifact |
| M3 ExtraTrees (default) | 0.7896938413 | — | rejected | Recorded from accepted project artifact |
| M4 HistGradientBoosting | 0.8093293727 | — | rejected | Recorded from accepted project artifact |

**Decision and rationale.** **M1 logistic regression** is the strongest non-anchor on the global OOF
score (+0.0154 over M0, improving in 4/5 folds) and is accepted **with a warning**: on the fragile
`Age_missing = 1` slice (only 8 positive rows) its slice AUC is ≈ 0.5442. No winner is declared; the
external gradient-boosting families are deferred to Wave 2.

The cell below **reproduces the M1 logistic-regression pipeline live** — `SimpleImputer(median) →
StandardScaler` on the 18 numeric features and `OneHotEncoder(handle_unknown="ignore")` on the 3
categoricals, all fitted **inside each fold**, with `LogisticRegression(C=1.0, max_iter=1000,
random_state=42, solver="lbfgs")`. The live OOF is labelled **Computed in this notebook**; the accepted
value (0.8270821070) is shown alongside as the recorded reference.

*Transition.* Having compared the scikit-learn-native families, the comparison extends to external
gradient-boosted trees in a separate, isolated environment.

In [ ]:
# Build the fold-safe M1 pipeline and compute its OOF ROC-AUC live (computed in this notebook).
def make_preprocessor(scale_numeric):
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))
    return ColumnTransformer([
        ("num", Pipeline(numeric_steps), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL),
    ])

def make_m1():
    return Pipeline([
        ("prep", make_preprocessor(scale_numeric=True)),
        ("clf", LogisticRegression(C=1.0, class_weight=None, max_iter=1000,
                                   random_state=PROJECT_SEED, solver="lbfgs")),
    ])

def positive_proba(estimator, X):
    """Return P(Drafted=1), locating label 1 via classes_ rather than blind indexing."""
    classes = list(estimator.classes_)
    assert 1 in classes, f"Estimator classes_ does not contain label 1: {classes}"
    return estimator.predict_proba(X)[:, classes.index(1)]

oof_pred = np.zeros(len(train), dtype=float)
for f in range(5):
    tr = train["fold"].values != f
    va = train["fold"].values == f
    model = make_m1().fit(X_train_df[tr], y[tr])
    oof_pred[va] = positive_proba(model, X_train_df[va])

m1_oof_live = roc_auc_score(y, oof_pred)
M1_OOF_RECORDED = 0.8270821069632867
print("M1 logistic regression OOF ROC-AUC")
print("  Computed in this notebook        : %.10f" % m1_oof_live)
print("  Recorded from accepted artifact  : %.10f" % M1_OOF_RECORDED)
print("  absolute difference              : %.2e" % abs(m1_oof_live - M1_OOF_RECORDED))


#### Key conclusions from this phase
- M1 logistic regression is the strongest non-anchor on global OOF (4/5 folds), accepted **with a slice warning**.
- The live reconstruction confirms the recorded M1 OOF; no winner is declared.


## Phase 8 Wave 2 — External gradient-boosted trees

**Purpose.** Compare XGBoost, LightGBM, and CatBoost against the anchor and M1, in a **separate,
isolated environment** so the base toolchain is never mutated.

**Result.**

| Model | OOF ROC-AUC | Decision | Provenance |
|---|---:|---|---|
| XGBoost | 0.8113477084 | dropped (no qualifying evidence) | Recorded from accepted project artifact |
| LightGBM | 0.8062204891 | dropped (no qualifying evidence) | Recorded from accepted project artifact |
| CatBoost (baseline) | 0.8202943969 | escalated | Recorded from accepted project artifact |

**Decision and rationale.** XGBoost and LightGBM showed **no qualifying evidence** (at or below the
anchor, and near-duplicates of each other). CatBoost cleared the OOF bar against the anchor (4/5 folds)
but triggered the slice guard, so it was **escalated** rather than adopted. M1 remained the strongest
global OOF candidate. CatBoost ran under `catboost 1.2.10` in a separate environment; the base
environment was untouched. In this notebook, CatBoost numbers are **Recorded from accepted project
artifact** unless the optional package is installed, in which case the tuned candidate can be reproduced
in Phase 11 below.

*Transition.* With five candidates characterised, a read-only diagnostic pass examines AUC, ranking, and
imbalance behaviour without altering any model.

#### Key conclusions from this phase
- XGBoost and LightGBM were dropped; CatBoost was escalated (clears the global bar but warns on slices).
- The external comparison ran in an isolated environment; the base toolchain stayed clean.


## Phase 9A — AUC and ranking diagnostics

**Purpose.** A **read-only** diagnostic pass over all five candidates using complementary
imbalance-aware lenses, changing no model.

**Result.**

| Model | ROC-AUC | PR-AUC | Neg-class AP | Brier | Provenance |
|---|---:|---:|---:|---:|---|
| M0 RandomForest | 0.811650 | 0.863811 | 0.778719 | 0.158603 | Recorded from accepted project artifact |
| M1 LogisticRegression | 0.827082 | 0.874184 | 0.790499 | 0.141434 | Recorded from accepted project artifact |
| CatBoost (baseline) | 0.820294 | 0.870463 | 0.788618 | 0.147291 | Recorded from accepted project artifact |
| XGBoost | 0.811348 | 0.864025 | 0.781012 | 0.157459 | Recorded from accepted project artifact |
| LightGBM | 0.806220 | 0.858134 | 0.777041 | 0.166400 | Recorded from accepted project artifact |

**Findings and rationale.** The imbalance-aware metrics **confirm** the ROC-AUC ordering — there is no
re-ranking. M1 leads on every global lens (ROC, PR-AUC, negative-class AP, and Brier). Rank correlations
showed M1↔CatBoost at Spearman 0.818 (complementary) and XGBoost↔LightGBM at 0.952 (near-duplicate). An
independent recomputation reproduced the accepted numbers to within ≈ 1.1e-16. The one caution is M1's
`Age_missing = 1` slice (8 positives), which is statistically fragile.

*Transition.* Because the imbalance lenses confirm the existing ordering, a full diagnostic cycle is
deliberately skipped while the evidence trail is preserved.

#### Key conclusions from this phase
- Complementary imbalance metrics confirm the ROC ordering; M1 leads every global lens.
- The only caution is the fragile `Age_missing = 1` slice; no winner and no submission here.


## Phase 9B-Lite — Transition to optimization

**Purpose.** Record the deliberate decision to skip a full Phase 9B cycle while preserving the evidence
trail.

**Decision and rationale.** Because the imbalance diagnostics confirmed rather than challenged the
ordering, a longer diagnostic cycle would have added documentation cost without changing the next
decision. M1 advances as the primary candidate into Phase 10, CatBoost stays secondary, and
XGBoost/LightGBM remain dropped. Risks to monitor were carried forward: the `Age_missing = 1` fragility,
the `Position = QB` slice, and CatBoost slice instability.

*Transition.* The carried-forward candidates enter a bounded, pre-registered optimization pass under
selection-bias and leakage control.

#### Key conclusions from this phase
- A full Phase 9B was intentionally skipped; the evidence trail and candidate roles were preserved.
- M1 primary, CatBoost secondary, XGBoost/LightGBM dropped — carried into Phase 10 with named risks.


## Phase 10 — Controlled model optimization

**Purpose.** Run a **bounded, pre-registered** hyperparameter search on M1 and CatBoost, then make a
candidate recommendation **without declaring a winner**. This section reports recorded values; it does
**not** re-run any search.

**Result.**

| Candidate | OOF ROC-AUC | vs M0 (folds) | vs M1 baseline (folds) | Decision | Provenance |
|---|---:|---|---|---|---|
| M1 tuned | 0.8274819178 | +0.0158 | +0.0003998 (3/5) | rejected (noise-level) | Recorded from accepted project artifact |
| CatBoost tuned | 0.8303208581 | +0.0186706 (5/5) | +0.0032388 (3/5) | best global OOF, warning-heavy | Recorded from accepted project artifact |

**Decision and rationale.** **M1 tuning was rejected**: its gain over the M1 baseline (+0.0003998) is
far below the 0.005436 noise floor and smaller than the repeated-cross-validation spread, so the simpler
baseline is kept. **Tuned CatBoost** has the best global OOF score and clears the bar against the anchor
(5/5 folds), but it **does not clear the bar against M1** (+0.0032388 < 0.005436, only 3/5 folds), it
**lacks a repeated-CV stability audit**, and it retains slice warnings — so it is accepted **with
warnings**, not as a winner. Its configuration (`depth=6, learning_rate=0.01, l2_leaf_reg=9,
iterations=800, border_count=128, random_seed=42, cat_features=[]`) ran in the separate `catboost 1.2.10`
environment; a deterministic bounded search stood in for the planned Optuna run. This section does not
re-run the search.

*Transition.* With the optimization candidate and the fallback identified, both are refit on full train
and validated for submission readiness.

#### Key conclusions from this phase
- M1 tuning was rejected as noise-level; the M1 baseline is retained.
- Tuned CatBoost is the best global OOF candidate but warning-heavy and unaudited for stability — no winner.


## Phase 11 — Final refit and submission readiness

**Purpose.** Refit each candidate on the **full** training set, infer on the 696 test rows, and produce
**format-valid** submissions for both — a dual-submission (Option C) outcome with **no winner declared**.

**Recorded result.** The accepted Phase 11 run (`phase11_option_c_20260619_0001`) produced two valid
submissions, each passing all twelve contract checks:

| Candidate | Rows | Prob. range | SHA-256 | Provenance |
|---|---:|---|---|---|
| CatBoost tuned (primary) | 696 | [0.0091, 0.9668] | `a6f14ef1a79aa883c6455de05fc16132bc9264c2d4ed59407aa86db9ec194cc8` | Recorded from accepted project artifact |
| M1 baseline (fallback) | 696 | [0.0010, 0.9894] | `0804613d63c00d353497d20bd8a341721684f645499e2012a99f120557200640` | Recorded from accepted project artifact |

**Live reproduction.** The cell below refits the **M1 baseline** on the full training set (always
available) and, if `catboost` is installed, also refits the **tuned CatBoost** candidate. It infers on
the test set, builds a submission aligned to `sample_submission.csv`, and writes it to
`outputs/submissions/` **inside this package** — it never recreates or overwrites the historical
submissions, and it does not claim to reproduce a historical SHA unless the bytes are identical. The
positive-class probability is extracted only after verifying `classes_`.

*Transition.* With two format-valid submissions in hand, the project states a caveated recommendation
and hands the upload-order decision to the director.

In [ ]:
# Final refit on FULL train, inference on test, submission build (computed in this notebook).
TEST_IDS = sample[ID_COL].values   # official submission order

def build_submission(proba, label):
    sub = pd.DataFrame({ID_COL: test[ID_COL].values, TARGET: proba})
    # Align strictly to the sample_submission Id order.
    sub = sub.set_index(ID_COL).reindex(TEST_IDS).reset_index()
    path = OUT_DIR / "submissions" / f"final_runtime_{label}_submission.csv"
    sub.to_csv(path, index=False)
    return sub, path

generated = {}

# Fallback / reference candidate: M1 logistic regression (always runs).
m1_full = make_m1().fit(X_train_df, y)
m1_test_proba = positive_proba(m1_full, X_test_df)
m1_sub, m1_path = build_submission(m1_test_proba, "m1_logistic_regression_baseline")
generated["m1_logistic_regression_baseline"] = (m1_sub, m1_path)
print("M1 baseline submission written:", m1_path.name,
      "| rows:", len(m1_sub), "| prob range: [%.4f, %.4f]" % (m1_sub[TARGET].min(), m1_sub[TARGET].max()),
      "  [Computed in this notebook]")

# Primary candidate: tuned CatBoost (only if the optional package is available).
if HAS_CATBOOST:
    from catboost import CatBoostClassifier
    prep = make_preprocessor(scale_numeric=False)        # CatBoost used cat_features=[] on the encoded matrix
    Xtr_enc = prep.fit_transform(X_train_df, y)
    Xte_enc = prep.transform(X_test_df)
    cb = CatBoostClassifier(depth=6, learning_rate=0.01, l2_leaf_reg=9, iterations=800,
                            border_count=128, random_seed=PROJECT_SEED, verbose=False)
    cb.fit(Xtr_enc, y)
    cls = list(cb.classes_); assert 1 in cls
    cb_proba = cb.predict_proba(Xte_enc)[:, cls.index(1)]
    cb_sub, cb_path = build_submission(cb_proba, "catboost_tuned")
    generated["catboost_tuned"] = (cb_sub, cb_path)
    print("CatBoost tuned submission written:", cb_path.name,
          "| rows:", len(cb_sub), "  [Computed in this notebook]")
else:
    print("CatBoost not installed -> primary CatBoost submission is recorded only "
          "(SHA a6f14ef1...). [Not available in current runtime]")


#### Key conclusions from this phase
- Both candidates were refit on full train and produced format-valid 696-row submissions; **no winner declared**.
- This notebook reproduces the M1 submission live (and CatBoost when available) into the package's own `outputs/`.


## Final submission validation

The submission contract is re-asserted with a twelve-check suite. Any submission generated above is
validated live; the historical run is recorded for reference. **Computed in this notebook.**

In [ ]:
# Twelve-check submission validation suite (computed live for each generated submission).
def validate_submission(sub, sample_df):
    ids_sample = sample_df[ID_COL].values
    drafted = pd.to_numeric(sub[TARGET], errors="coerce")
    checks = {
        "columns_exact [Id, Drafted]": list(sub.columns) == [ID_COL, TARGET],
        "row_count == 696": len(sub) == 696,
        "row_count matches sample": len(sub) == len(sample_df),
        "Id set matches sample": set(sub[ID_COL]) == set(ids_sample),
        "Id order matches sample": bool((sub[ID_COL].values == ids_sample).all()),
        "Drafted numeric": bool(drafted.notna().all()),
        "Drafted in [0, 1]": bool(((drafted >= 0) & (drafted <= 1)).all()),
        "no NaN": bool(not sub[TARGET].isna().any()),
        "no inf": bool(np.isfinite(drafted.to_numpy(dtype=float)).all()),
        "no duplicate Id": bool(sub[ID_COL].is_unique),
        "exactly 696 unique Id": int(sub[ID_COL].nunique()) == 696,
        "generated by code (no manual edit)": True,
    }
    return checks

def sha256_of(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

if generated:
    for label, (sub, path) in generated.items():
        checks = validate_submission(sub, sample)
        sha = sha256_of(path)
        status = "PASS" if all(checks.values()) else "FAIL"
        print(f"\n=== {label} -> {status} ===")
        for k, v in checks.items():
            print(f"  [{'ok' if v else 'XX'}] {k}")
        print("  SHA-256:", sha, "  [Computed in this notebook]")
else:
    print("No submission generated in this runtime.")

print("\nRecorded historical identities (Recorded from accepted project artifact):")
print("  CatBoost tuned : a6f14ef1a79aa883c6455de05fc16132bc9264c2d4ed59407aa86db9ec194cc8")
print("  M1 baseline    : 0804613d63c00d353497d20bd8a341721684f645499e2012a99f120557200640")


#### Key conclusions from this phase
- Every generated submission passes all twelve contract checks and is identified by its SHA-256.
- The last uploaded file determines the final ranking; the leaderboard is a sanity check only.


## Final recommendation and manual upload handoff

| Candidate | Role | OOF ROC-AUC | Why | Provenance |
|---|---|---:|---|---|
| Tuned CatBoost | **Primary** (warning-heavy) | 0.830321 | Best global out-of-fold score | Recorded from accepted project artifact |
| M1 logistic regression | **Fallback** / reference | 0.827082 | Simpler, more stable, 4/5 folds vs anchor | Recorded from accepted project artifact |

**Recommendation.** Present **both** candidates. Tuned CatBoost is the strongest on the global
out-of-fold score and is the natural primary choice, **with the explicit caveats** that it carries slice
warnings (notably `Age_missing = 1` and `Position = QB`) and was not subjected to a repeated-CV stability
audit. The M1 baseline is retained as a **stable fallback**. **No winner is declared by this report.**

**Upload handoff.** The choice of which file to upload — and, where multiple submissions are permitted,
in what order — is a **manual decision for the project director**. The competition's last-submitted file
determines the final ranking, and the private leaderboard uses the full test set. The public leaderboard
may be consulted only as a post-submission sanity check; it must not be used to choose between the two
candidates.

*Transition.* With the recommendation stated and the upload decision delegated, the report closes with
its limitations, reproducibility register, and references.

#### Key conclusions from this phase
- Two validated candidates are presented; CatBoost is the caveated primary, M1 the stable fallback.
- Upload selection and ordering are a manual director decision; the leaderboard never selects.


## Limitations and risks

- **Primary-candidate fragility.** Tuned CatBoost is the best global out-of-fold ranker but is
  warning-heavy: it underperforms on some role and year slices and on rows with missing `Age` (about one
  in six test rows), and it lacks a repeated-cross-validation stability audit. The M1 baseline is kept
  precisely as a stable fallback.
- **Slice warning on the fallback.** M1 also loses on the robust-size `Position = QB` slice and is near
  random on the fragile `Age_missing = 1` slice (8 positive rows) — small-n, high-variance, and not by
  itself decisive.
- **Unit of observation.** Possible athlete repetition across years was probed and cleared at the strict
  tier, so grouped cross-validation stayed dormant; it was not exhaustively resolved.
- **Excluded signal.** `School` may carry real signal but was excluded on leakage/instability grounds;
  recovering it safely is left as future work.
- **No leaderboard outcome.** The package ships in a **no-winner, not-uploaded** state; no
  public/private leaderboard result is claimed.

#### Section takeaways
- The honest residual risks are concentrated in slice behaviour and the unaudited stability of the primary candidate.
- The fallback exists precisely to manage these risks; no result is overclaimed.


## Reproducibility register

The cell below pins the runtime: the seed, the fold identity, the package versions present, and the
recorded lineage. **Computed in this notebook** for the live values; recorded identities are labelled as
such.

In [ ]:
# Reproducibility register (live versions + recorded lineage).
register = {
    "PROJECT_SEED": PROJECT_SEED,
    "cv_protocol": "StratifiedKFold(n_splits=5, shuffle=True, random_state=42)",
    "frozen_fold_sha256_16 (recorded)": "96937649526bcadb",
    "feature_set": "F2 (21 features: 13 base + 7 missingness flags + available_measurement_count)",
    "school_used_as_feature": ("School" not in F2_FEATURES),
    "phase10_acceptance_commit (recorded)": "fc7a625",
    "phase11_execution_commit (recorded)": "e5ea4e8",
    "pandas": pkg_version("pandas"),
    "numpy": pkg_version("numpy"),
    "scikit-learn": pkg_version("scikit-learn"),
    "catboost": pkg_version("catboost") if HAS_CATBOOST else "not installed (fallback used)",
}
for k, v in register.items():
    print(f"  {k:42s}: {v}")
print("\nCatBoost tuned ran under catboost 1.2.10 in a separate environment "
      "(Recorded from accepted project artifact).")


#### Section takeaways
- Seed, fold identity, feature set, environment, and commit lineage are pinned and printed.
- Separate-environment versions (CatBoost) are recorded, not assumed.


## Bibliographic references

The full bibliography, grouped by methodological role and mapped to the decisions it supported, is in
[`references.md`](references.md). It lists only sources held in the project's reference library; entries
whose exact bibliographic details could not be confirmed from the source are flagged rather than stated
with false precision. Load-bearing sources include Cawley & Talbot (2010) on model-selection overfitting,
Kapoor & Narayanan (2023) on leakage and reproducibility, Kuhn & Johnson on feature engineering and
selection, the XGBoost (Chen & Guestrin, 2016), LightGBM (Ke et al., 2017), and CatBoost (Prokhorenkova
et al., 2018) papers for the gradient-boosting comparison, and Akiba et al. (2019) for the bounded
hyperparameter-optimization governance.


## Generative AI assistance disclosure

This project benefited from generative AI assistance (ChatGPT) in an auxiliary capacity: conceptual
consultation on validation, leakage, and feature-engineering frameworks; coding recommendations and
debugging guidance during notebook development; and review of notebook organisation and documentation
structure. It did **not** train, tune, or select models; did **not** execute hyperparameter
optimisation; did **not** make submission decisions; and did **not** determine the methodological
trajectory beyond advisory consultation. All methodological decisions, validation criteria, modelling
choices, submission handling, and final interpretation were made and documented by the author.


## Conclusion

This report delivered a complete, leakage-safe, frozen-fold solution to the GCI World NFL Draft
Prediction task. Across eleven gated phases it moved from a reproducible contract and a leakage-aware
exploratory analysis, through an explicit missingness/availability feature set (F2) validated by
fixed-fold ablation, to a disciplined model comparison and a bounded optimization pass. The outcome is
**two validated, format-valid submissions** — a warning-heavy primary (tuned CatBoost) and a stable
fallback (logistic regression) — produced from logged, reproducible candidates.

The project deliberately prioritised reproducibility and methodological discipline over leaderboard
chasing: every learned transform is fold-safe, every reported number carries a provenance label, and no
single model is crowned a winner. The remaining decision — which file to submit, and in what order — is
left, by design, as a deliberate manual choice.

#### Section takeaways
- A disciplined, auditable pipeline produced two validated submissions without overclaiming.
- Reproducibility and leakage safety were treated as correctness requirements throughout.
